In [2]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

# FP16 matmul - this hits Tensor Cores directly
import time
torch.cuda.init()

size = 8192
a = torch.randn(size, size, dtype=torch.float16, device='cuda')
b = torch.randn(size, size, dtype=torch.float16, device='cuda')

# warmup
torch.mm(a, b)
torch.cuda.synchronize()

# benchmark
start = time.time()
for _ in range(20):
    c = torch.mm(a, b)
torch.cuda.synchronize()
elapsed = time.time() - start

ops = 20 * 2 * size**3
tflops = ops / elapsed / 1e12
print(f'FP16 TFLOPS: {tflops:.1f}')
print(f'Expected with Tensor Cores: ~90-112 TFLOPS')
print(f'Expected WITHOUT Tensor Cores: ~14 TFLOPS')
print(f'Tensor Cores alive: {tflops > 40}')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla V100-SXM2-16GB
FP16 TFLOPS: 90.9
Expected with Tensor Cores: ~90-112 TFLOPS
Expected WITHOUT Tensor Cores: ~14 TFLOPS
Tensor Cores alive: True
